# 04 · QKI eCLIP from RBP-ARK and rbp-maps splice maps (panels d, e)

Fifth of six notebooks (00 screen and locus → 01 download → 02 rMATS → 03 panel c → **04 eCLIP and rbp-maps** → 05 optional window maps). Inputs: the rMATS skipped-exon tables from notebook 02 (or ENCODE's release). It follows the rbp-maps [notebook walkthrough](https://github.com/YeoLab/rbp-maps/blob/master/documentation/notebook_walkthrough.ipynb): get read-2 BAMs, make strand-specific RPM bigWigs, load them as `ReadDensity` objects, and build a `Map.SkippedExon` splice map.

Panels d and e ask whether PICALM's two exons follow QKI's genome-wide rule. A splice map averages input-normalized eCLIP signal around cassette exons that are more included after knockdown (exons the RBP normally represses), more excluded (exons it activates), or unchanged (background). Each PICALM exon's own row of the same normalized matrix shows where its signal sits: upstream of the 3′ splice site for exon 13 (QKI represses it), downstream of the 5′ splice site for the 24-nt exon (QKI promotes it).

| Section | Step |
| --- | --- |
| 1 | Download the 7 eCLIP BAMs of the two QKI datasets in the splice maps, from RBP-ARK |
| 2 | Build the event lists for the two QKI knockdowns: filter rMATS, lift hg19 → GRCh38 |
| 3 | Check read strand and make RPM bigWigs |
| 4 | Splice maps with rbp-maps: both eCLIP datasets against the HepG2 QKI knockdown |
| 5 | PICALM's own traces and a panel-d/e-style figure |

**Environment and time.** `encode-rbp-case-study` (rbp-maps, samtools, bedtools, bedGraphToBigWig, pyliftover). The BAMs total 3.7 GB. There are 4 maps (2 datasets × 2 replicates, both against the HepG2 knockdown); with the permutation test each takes about 10 minutes. `RBPMAPS_SIGTEST=zscore` takes about a minute per map.

## Accessing RBP-ARK data through Globus

RBP-ARK keeps every Skipper output on one Globus collection. Public files can be downloaded straight over HTTPS with no login; bulk copies go through Globus.

| | |
| --- | --- |
| Globus collection | **RBP-ARK**, `9cd9be17-cbed-4b34-ab76-704b2fd6ac7b` ([open in the Globus web app](https://app.globus.org/file-manager?origin_id=9cd9be17-cbed-4b34-ab76-704b2fd6ac7b&origin_path=%2F)) |
| HTTPS address of the same files | `https://g-f8db23.0ed28.75bc.data.globus.org/<path>` |

**Where things are.** `<version>/<run>` is the Skipper version and run folder. Since November 2026 every ENCODE3, ENCORE and ENCORE2 dataset on rbp-ark.com (gene search, dataset pages, BDBags) is Skipper `2.0.0/v2`, and so are these notebooks. Earlier runs (`1.99.0/v1`, `1.100.0/v1`) remain on the collection.

| Path on the collection | Contents |
| --- | --- |
| `/encode3/eclip/Skipper/<version>/<run>/output/` | ENCODE3 eCLIP datasets, named `<RBP>_<cell>_ENCSR…` |
| `/encore2/eclip/Skipper/<version>/<run>/output/` | ENCORE2 eCLIP datasets with 5000-series IDs (e.g. `QKI_K562_5007`) |
| `/encore/eclip/Skipper/<version>/<run>/output/` | ENCORE eCLIP datasets with 4000-series IDs |
| `/summary_data/` | summary tables (QC, window counts) |

Inside each `output/` folder, files are grouped by type, then named by dataset:

| Subfolder | Files |
| --- | --- |
| `reproducible_enriched_windows/` | `<dataset>.reproducible_enriched_windows.tsv.gz`, the enriched windows |
| `reproducible_enriched_re/` | enriched repetitive elements |
| `secondary_results/bams/dedup/genome_R2/` | deduplicated, read-2-only genome BAMs (ENCODE3) |
| `secondary_results/bams/dedup/genome/` | deduplicated genome BAMs (ENCORE) |
| `secondary_results/bams/raw/genome/` | alignments before deduplication |
| `secondary_results/bigwigs/scaled/plus/`, `.../minus/` | strand-specific signal tracks |
| `QC/`, `gene_sets/`, `homer/` | QC reports, gene set enrichment, motifs |

**Three ways to get files.**

1. **One dataset: BDBag.** Each dataset page on rbp-ark.com has a **Download BDBag** link. The bag's `fetch.txt` lists every file of that dataset with its HTTPS URL, size and path, and `manifest-sha256.txt` its checksum. Notebook 04 reads the same file list from the RBP-ARK API and downloads only the BAMs it needs.
2. **Many datasets: Globus web app.** Sign in at [app.globus.org](https://app.globus.org) with your institution or a free Globus ID, open the RBP-ARK collection (link above), select folders or files, and transfer them to a collection you can write to: your laptop via [Globus Connect Personal](https://www.globus.org/globus-connect-personal), or your institution's cluster or storage collection.
3. **Scripted: Globus CLI.**

```bash
pip install globus-cli
globus login                                   # once; opens a browser
RBPARK=9cd9be17-cbed-4b34-ab76-704b2fd6ac7b
OUT=/encode3/eclip/Skipper/2.0.0/v2/output

# list what is there
globus ls "$RBPARK:$OUT/secondary_results/bams/dedup/genome_R2/" | grep QKI

# copy one file to this computer (Globus Connect Personal must be running)
DEST=$(globus endpoint local-id)
globus transfer "$RBPARK:$OUT/secondary_results/bams/dedup/genome_R2/QKI_K562_ENCSR366YOG_IP_1.genome.Aligned.sort.dedup.R2.bam" \
    "$DEST:$HOME/rbpark/QKI_K562_ENCSR366YOG_IP_1.genome.Aligned.sort.dedup.R2.bam" --label "QKI eCLIP"

# or copy a whole folder
globus transfer --recursive "$RBPARK:$OUT/reproducible_enriched_windows/" "$DEST:$HOME/rbpark/windows/"
globus task list                               # follow progress
```

If the CLI asks for extra consent the first time it touches the collection, it prints the exact `globus session consent ...` command to run. The Globus CLI commands are shown for reference and were not run in testing; the BDBag and HTTPS routes were. Please copy only what you need: the public HTTPS endpoint is shared, and bulk sweeps have taken it offline.

**In this notebook:** section 1 downloads the 7 eCLIP BAMs of the two QKI datasets in the splice maps over HTTPS from the collection's public endpoint, using the file list (URL, size, SHA-256) from the RBP-ARK API. To copy them with Globus instead, transfer the `secondary_results/bams/dedup/` files listed in section 1 into `<data>/rbp_ark_eclip_skipper2/full/`; the notebook then checks and uses them.

In [ ]:
import hashlib
import json
import os
import random
import shutil
import subprocess
import sys
from collections import OrderedDict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysam
from IPython.display import Image, display

from maps import plot_map
import density.ReadDensity
import density.normalization_functions as norm
from density import Map

sys.path.insert(0, ".")
import cs3_utils as U

ROOT, DATA = U.folders()
RMATS_DIR = DATA / "rmats"
ECLIP_DIR = DATA / "rbp_ark_eclip_skipper2" / "full"
API_CACHE = DATA / "downloads" / "rbpark_api"
SIGNAL_DIR = DATA / "rbp_maps" / "bigwigs"
SOURCE = os.environ.get("SE_SOURCE", "rerun")          # "rerun" (notebook 02) or "encode" (ENCODE release)
EVENT_DIR = DATA / "rbp_maps" / "event_lists_{}".format(SOURCE)
MAP_DIR = DATA / "rbp_maps" / "maps_{}".format(SOURCE)
FIG_DIR = ROOT / "figures" / "panel_de"
for d in (ECLIP_DIR, API_CACHE, SIGNAL_DIR, EVENT_DIR, MAP_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

SIGTEST = os.environ.get("RBPMAPS_SIGTEST", "permutation")   # "zscore" for a quick look
VERIFY_SHA256 = True
THREADS = os.cpu_count() or 4

for tool in ("samtools", "bedtools", "bedGraphToBigWig"):
    assert shutil.which(tool), tool + " is not on PATH"
print("ROOT:", ROOT, "| DATA:", DATA, "| rMATS source:", SOURCE, "| test:", SIGTEST, "| maps:", U.MAP_PAIRS)

## 1. QKI eCLIP BAMs from RBP-ARK

RBP-ARK lists three QKI eCLIP datasets that pass QC. The splice maps use two of them; the third (ENCODE3 K562) gives windows only, in notebook 00. For each, the RBP-ARK API (`/api/experiments/<dataset>/`) lists every Skipper output file with its URL, size and SHA-256, the same list a dataset page's BDBag holds. This notebook downloads only the deduplicated genome BAMs (GRCh38) that rbp-maps needs.

| RBP-ARK dataset | Cell line | Project | BAMs used | Pairing |
| --- | --- | --- | --- | --- |
| QKI_K562_5007 | K562 | ENCORE2 | `IP_1`, `IP_2`, `IN_1`, `IN_2` | `IP_1` vs `IN_1`, `IP_2` vs `IN_2` |
| QKI_HepG2_ENCSR570WLM | HepG2 | ENCODE3 | `IP_1`, `IP_2`, `IN_1` (read 2 only) | both replicates vs the one input |

The ENCORE pairing is an assumption from the sample names; check the dataset page if it matters. Files already in `<data>/rbp_ark_eclip_skipper2/full/` with the right size and checksum are not downloaded again.

In [ ]:
def pick_bam(files, name, sample):
    """Returns the file entry of the deduplicated BAM of `sample` in dataset `name` from RBP-ARK's file list `files`,
    preferring the read-2-only BAM (genome_R2).
    """
    pattern = r"/bams/dedup/genome(?:_R2)?/{}_{}\.genome\.Aligned\.sort\.dedup(?:\.R2)?\.bam$".format(name, sample)
    hits = files[files["collection_path"].str.contains(pattern)]
    assert len(hits), "{}: no deduplicated BAM for {}".format(name, sample)
    return hits.sort_values("collection_path", key=lambda s: ~s.str.contains("genome_R2")).iloc[0]


def sha256(path, chunk=8 << 20):
    """Returns the SHA-256 checksum of the file at `path`, read `chunk` bytes at a time."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def download(row, out):
    """Downloads the file in RBP-ARK entry `row` to `out` (resuming a partial file), checks its size and SHA-256, indexes
    it and returns `out`.
    """
    ok = Path(str(out) + ".sha256ok")
    if not (out.exists() and out.stat().st_size == row["size"]):
        subprocess.check_call(["curl", "-L", "-sS", "--retry", "5", "-C", "-", "-o", str(out), row["url"]])
    assert out.stat().st_size == row["size"], out
    if VERIFY_SHA256 and row["sha256"] and not ok.exists():
        assert sha256(out) == row["sha256"], "{}: checksum mismatch; delete it and re-run".format(out.name)
        ok.write_text(row["sha256"] + "\n")
    if not Path(str(out) + ".bai").exists():
        subprocess.check_call(["samtools", "index", "-@", str(THREADS), str(out)])
    return out


bams = []
for name in dict.fromkeys(n for n, _ in U.MAP_PAIRS):     # the eCLIP datasets in the splice maps
    info = U.ECLIP[name]
    files = U.rbpark_files(name, API_CACHE)
    for sample in sorted({s for pair in info["reps"].values() for s in pair}):
        row = pick_bam(files, name, sample)
        local = ECLIP_DIR / Path(row["collection_path"]).name
        print("checking" if local.exists() else "downloading", local.name, "({:.2f} GB)".format(row["size"] / 1e9), flush=True)
        download(row, local)
        bams.append({"dataset": name, "rbp": info["rbp"], "cell": info["cell"], "sample": sample, "bam": str(local),
                     "read2_only_file": "genome_R2" in row["collection_path"], "size_GB": round(row["size"] / 1e9, 2)})
bams = pd.DataFrame(bams)
bams

## 2. Event lists: filter the rMATS tables, then lift hg19 → GRCh38

The rMATS events are in hg19 (the knockdown BAMs' assembly) and the eCLIP BAMs in GRCh38. Three sets per knockdown:

| Set | Rule |
| --- | --- |
| included upon knockdown | FDR < 0.1, P < 0.05, ΔPSI > 0.05 |
| excluded upon knockdown | FDR < 0.1, P < 0.05, ΔPSI < −0.05 |
| native cassette background | P > 0.5, mean PSI strictly between 0.05 and 0.95 in both knockdown and control |

There is no minimum read count; rows whose PSI is NA in every replicate of a condition cannot pass the background rule.

**One event per exon.** rMATS reports each form of an exon as its own event (PICALM exon 13 appears as 150-, 129- and 48-nt forms that share its 5′ splice site), and rbp-maps would then average the same intron signal once per form. Each set is therefore reduced with the rule of rbp-maps' own [`subset_rmats_junctioncountonly.py`](https://github.com/YeoLab/rbp-maps/blob/master/preprocessing_scripts/subset_rmats_junctioncountonly.py) (the step rbp-maps runs with `--auto_subset_rmats`): it merges events whose spans from the upstream exon's end to the downstream exon's start overlap (bedtools merge) and keeps the one with the highest mean inclusion junction count. rbp-maps leaves ties to `bedtools sort`, whose order differs between macOS and Linux, so `U.one_event_per_exon()` applies the same rule in Python and gives a tie to the event listed first; every computer then keeps the same events. For PICALM exon 13 that is the 150-nt form. The merge is broader than "forms of one exon": neighbouring cassette exons whose flanking introns overlap are also reduced to one event, and it ignores strand.

Each kept event's three exons are lifted with [pyliftover](https://github.com/konstantint/pyliftover) (UCSC hg19ToHg38 chain, downloaded on first use), as UCSC liftOver would: an event is kept only if all three exons lift to one chromosome, on the event's strand, with unchanged lengths. One extra guard: an event whose lifted exons come out of order (a negative intron, seen in segmental duplications) is dropped, because rbp-maps cannot build its regions. Lines are written back unchanged except for the coordinates, because rbp-maps reads rMATS rows by position.

PICALM exon 13 should be in the included set for both QKI knockdowns, and the 24-nt exon in the excluded set.

In [ ]:
from pyliftover import LiftOver

LIFT = LiftOver("hg19", "hg38")
INTERVALS = [("exonStart_0base", "exonEnd"), ("upstreamES", "upstreamEE"), ("downstreamES", "downstreamEE")]
SETS = ["included-upon-knockdown", "excluded-upon-knockdown", "native-cassette-background"]


def se_path(key):
    """Returns the rMATS skipped-exon table for knockdown `key`: notebook 02's rerun, or ENCODE's release when
    SE_SOURCE=encode.
    """
    if SOURCE == "rerun":
        return RMATS_DIR / "rerun" / key / "SE.MATS.JC.txt"
    return RMATS_DIR / "encode_release" / key / "SE.MATS.JunctionCountOnly.txt"


def total_reads(text):
    """Returns the sum of the junction counts in rMATS field `text`, skipping NA."""
    return sum(int(float(x)) for x in str(text).split(",") if x not in ("NA", ""))


def mean_psi(text):
    """Returns the mean of the PSI values in rMATS field `text` (NaN if all are NA)."""
    vals = [float(x) for x in str(text).split(",") if x not in ("NA", "")]
    return np.mean(vals) if vals else np.nan


def classify(table):
    """Returns boolean masks of the three event sets (included, excluded, native background) for rMATS table `table`.
    """
    psi_kd, psi_ctl = table["IncLevel1"].map(mean_psi), table["IncLevel2"].map(mean_psi)
    keep = ~U.gene_ids(table).isin(U.EXCLUDED_GENES)          # genes left out of every event list (cs3_utils.py)
    sig = keep & (table["FDR"] < 0.1) & (table["PValue"] < 0.05)
    return {
        "included-upon-knockdown": sig & (table["IncLevelDifference"] > 0.05),
        "excluded-upon-knockdown": sig & (table["IncLevelDifference"] < -0.05),
        "native-cassette-background": keep & (table["PValue"] > 0.5) & psi_kd.between(0.05, 0.95, inclusive="neither")
                                      & psi_ctl.between(0.05, 0.95, inclusive="neither"),
    }


def lift_point(chrom, pos):
    """Returns (chrom, position, strand) of hg19 position `pos` on `chrom` in GRCh38, or None unless it maps to exactly
    one place.
    """
    hits = LIFT.convert_coordinate(chrom, int(pos))
    return hits[0][:3] if hits and len(hits) == 1 else None



def exons_in_order(f, col):
    """Returns True if the exons of rMATS line `f` are in order (upstream < cassette < downstream), as rbp-maps needs;
    after liftover, segmental duplications can scatter them (seen for one NBPF11 event). `col` maps column names to
    indices.
    """
    return (int(f[col["upstreamEE"]]) <= int(f[col["exonStart_0base"]])
            and int(f[col["exonEnd"]]) <= int(f[col["downstreamES"]]))


def lift_fields(fields, col):
    """Returns the rMATS line `fields` with its three exons lifted to GRCh38 as UCSC liftOver would (each whole, on one
    chromosome and the event's strand, length unchanged), or None. `col` maps column names to indices.
    """
    chrom = fields[col["chr"]]
    out, chroms = list(fields), set()
    for start_name, end_name in INTERVALS:
        start, end = int(fields[col[start_name]]), int(fields[col[end_name]])
        a, b = lift_point(chrom, start), lift_point(chrom, end - 1)
        if a is None or b is None or a[0] != b[0] or a[2] != "+" or b[2] != "+":
            return None
        if (b[1] + 1) - a[1] != end - start:
            return None
        chroms.add(a[0])
        out[col[start_name]], out[col[end_name]] = str(a[1]), str(b[1] + 1)
    if len(chroms) != 1:
        return None
    out[col["chr"]] = chroms.pop()
    return out if exons_in_order(out, col) else None


def build_event_lists(key):
    """Writes the three GRCh38 event lists of knockdown `key` (filter, one event per exon, liftover); returns the counts
    after each step and the files.
    """
    path = se_path(key)
    lines = path.read_text().splitlines()
    header = lines[0].split("\t")
    col = {name: header.index(name) for name in header}   # index() returns the first of the two ID columns
    table = pd.read_csv(path, sep="\t")
    assert len(table) == len(lines) - 1
    masks = classify(table)
    counts, written = {}, {}
    for set_name in SETS:
        picked = np.flatnonzero(masks[set_name].to_numpy())
        rows = [lines[i + 1].split("\t") for i in picked]
        keep_ids = set(U.one_event_per_exon(rows, col))  # one event per exon, by rbp-maps' rule
        kept = [r for r in rows if r[0] in keep_ids]      # original lines, so rbp-maps reads them unchanged
        lifted = [f for f in (lift_fields(r, col) for r in kept) if f is not None]
        out = EVENT_DIR / "{}.GRCh38.{}.txt".format(key, set_name)
        text = lines[0] + "\n" + "".join("\t".join(f) + "\n" for f in lifted)
        if not (out.exists() and out.read_text() == text):   # unchanged lists keep their date, so finished maps are reused
            out.write_text(text)
        counts[set_name] = (len(rows), len(kept), len(lifted))
        written[set_name] = out
    return counts, written


def picalm_sets(files):
    """Returns the event set each PICALM exon is in, given the event-list `files` (GRCh38 coordinates)."""
    found = {}
    for set_name, path in files.items():
        t = pd.read_csv(path, sep="\t")
        for exon, (c, s, a, b) in U.PICALM_GRCH38.items():
            if ((t["chr"] == c) & (t["strand"] == s) & (t["exonStart_0base"] == a) & (t["exonEnd"] == b)).any():
                found[exon] = set_name
    return found


event_files, rows = {}, []
for key in U.KNOCKDOWNS:
    counts, event_files[key] = build_event_lists(key)
    where = picalm_sets(event_files[key])
    for set_name, (n_hg19, n_subset, n_grch38) in counts.items():
        rows.append({"knockdown": key, "set": set_name, "hg19": n_hg19, "one per exon": n_subset, "GRCh38": n_grch38,
                     "PICALM exons in set": ", ".join(e for e, s in where.items() if s == set_name)})
event_counts = pd.DataFrame(rows)
event_counts.to_csv(ROOT / "results" / "rbp_maps_event_counts_{}.tsv".format(SOURCE), sep="\t", index=False)
event_counts

Expected from ENCODE's release (`SE_SOURCE=encode`), GRCh38 events (included / excluded / background): QKI K562 about 17 / 10 / 9,500; QKI HepG2 about 307 / 105 / 9,000. These counts come from the window-based maps, which lift only the cassette exon; lifting all six coordinates, as here, can drop a few more.

## 3. Read strand and RPM bigWigs

rbp-maps needs reads that map to the same strand as the RNA. For paired-end eCLIP that is read 2, which is why ENCODE3 BAMs from RBP-ARK are already read-2-only (`genome_R2`). The ENCORE and ENCORE2 BAMs are deduplicated genome alignments; if one still contains read 1 of a pair, the cell below keeps read 2 only (`samtools view -f 128`), as the walkthrough does.

Then it measures strand from the data: the fraction of reads over background cassette exons that lie on the exon's own strand. Sense reads (fraction > 0.5) mean `direction="f"`; antisense reads mean `"r"`. Chromosome sizes for `bedGraphToBigWig` come from the BAM header, so they always match the alignments.

In [ ]:
def has_read1(path, n=200_000):
    """Returns True if the BAM at `path` still holds read 1 of paired-end reads (checks the first `n` reads)."""
    with pysam.AlignmentFile(path) as bam:
        for i, read in enumerate(bam.fetch(until_eof=True)):
            if read.is_paired and read.is_read1:
                return True
            if i >= n:
                return False
    return False


def read2_only(path):
    """Returns a BAM with read 2 only: `path` itself if it has no read 1, else a .R2only.bam copy, made once."""
    if not has_read1(path):
        return path
    r2 = path.replace(".bam", ".R2only.bam")
    if not Path(r2).exists():
        subprocess.check_call(["samtools", "view", "-@", str(THREADS), "-f", "128", "-b", "-o", r2, path])
        subprocess.check_call(["samtools", "index", r2])
    return r2


def sense_fraction(path, events_file, n_events=3000):
    """Returns the fraction of reads in the BAM at `path` that lie on the same strand as the cassette exon they overlap,
    and how many reads were checked, over up to `n_events` events from `events_file`.
    """
    table = pd.read_csv(events_file, sep="\t").sample(frac=1, random_state=0).head(n_events)
    sense = total = 0
    with pysam.AlignmentFile(path) as bam:
        contigs = set(bam.references)
        for _, e in table.iterrows():
            if e["chr"] not in contigs:     # events lifted onto alt/random contigs the eCLIP BAM lacks
                continue
            for read in bam.fetch(e["chr"], int(e["exonStart_0base"]), int(e["exonEnd"])):
                if read.is_secondary or read.is_supplementary or read.is_unmapped:
                    continue
                total += 1
                sense += (read.is_reverse == (e["strand"] == "-"))
    return sense / max(total, 1), total


def chrom_sizes(path):
    """Writes the chromosome sizes from the header of the BAM at `path` (for bedGraphToBigWig) and returns the file.
    """
    out = SIGNAL_DIR / "{}.chrom.sizes".format(Path(path).name)
    with pysam.AlignmentFile(path) as bam:
        out.write_text("".join("{}\t{}\n".format(n, l) for n, l in zip(bam.references, bam.lengths)))
    return out


signal = []
for _, row in bams.iterrows():
    bam = read2_only(row["bam"])
    background = event_files["{}_{}".format(row["rbp"], row["cell"])]["native-cassette-background"]
    frac, n = sense_fraction(bam, background)
    direction = "f" if frac > 0.5 else "r"
    pos = str(SIGNAL_DIR / Path(bam).name.replace(".bam", ".norm.pos.bw"))
    neg = str(SIGNAL_DIR / Path(bam).name.replace(".bam", ".norm.neg.bw"))
    plot_map.ensure_density_bigwigs(bam=bam, pos_bw=pos, neg_bw=neg, genome_file=str(chrom_sizes(bam)),
                                    direction=direction, makebigwigfiles_cmd=None,
                                    makebigwigfiles_extra_args="", makebigwigfiles_workdir=str(SIGNAL_DIR / "work"))
    signal.append({**row.to_dict(), "bam_used": bam, "reads_checked": n, "sense_fraction": round(frac, 3),
                   "direction": direction, "pos_bw": pos, "neg_bw": neg})
signal = pd.DataFrame(signal)
signal[["dataset", "sample", "reads_checked", "sense_fraction", "direction"]]

## 4. Splice maps

One map per eCLIP dataset, knockdown and replicate: IP vs its size-matched input, normalized with `per_region_subtract_and_normalize` (IP minus input, then each event scaled to sum to 1; rbp-maps normalization level 1), 50 nt of exon and 300 nt of intron per region, outlier removal keeping 95% of events.

Both datasets are mapped against the HepG2 QKI knockdown (`MAP_PAIRS` in `cs3_utils.py`), the K562 eCLIP included: QKI knockdown in K562 is the weaker knockdown (log2FC −0.85 vs −3.36 in HepG2), so the HepG2 knockdown gives the stronger map, as in the window-based maps.

**Rerunning.** With the permutation test, a map whose outputs are complete and newer than its event lists is read back from disk instead of being recomputed, so a run that stops partway resumes where it stopped (event lists are only rewritten when their content changes). `RBPMAPS_RECOMPUTE=1` recomputes every map. Each map closes its bigWig and BAM files when done; holding them open across many maps hits macOS's limit of 256 open files.

In [ ]:
def read_density(row):
    """Returns an rbp-maps ReadDensity for one eCLIP sample, from `row`'s bigWigs and BAM."""
    return density.ReadDensity.ReadDensity(pos=row["pos_bw"], neg=row["neg_bw"], bam=row["bam_used"])


def splice_map(name, kd_key, rep):
    """Builds the rbp-maps splice map of eCLIP dataset `name`, replicate `rep`, against the event lists of knockdown
    `kd_key`; returns the map and its png.
    """
    ip_sample, input_sample = U.ECLIP[name]["reps"][rep]
    sub = signal[signal["dataset"] == name].set_index("sample")
    ip, inp = read_density(sub.loc[ip_sample]), read_density(sub.loc[input_sample])
    files = event_files[kd_key]
    annotations = OrderedDict((str(files[s]), "rmats") for s in SETS)
    tests, background = [str(files[SETS[0]]), str(files[SETS[1]])], str(files[SETS[2]])

    out = map_png(name, kd_key, rep)
    out.parent.mkdir(parents=True, exist_ok=True)
    m = Map.SkippedExon(ip, inp, str(out), norm.per_region_subtract_and_normalize, annotations,
                        exon_offset=50, intron_offset=300, min_density_threshold=0, conf=0.95)
    m.create_matrices()
    m.normalize_matrix()
    m.create_lines()
    random.seed(U.PERMUTATION_SEED)              # rbp-maps' background subsets: the same every run
    m.set_background_and_calculate_significance(tests, background, SIGTEST)
    m.write_intermediates_to_csv()
    m.plot(tests)
    plt.close("all")
    # Close the bigWig and BAM handles: each ReadDensity keeps three files open, and many maps exceed macOS's
    # default limit of 256 open files ("Too many open files" at the last map).
    for rd in (ip, inp):
        for handle in (getattr(rd, "pos", None), getattr(rd, "neg", None), getattr(rd, "bam", None)):
            if handle is not None:
                handle.close()
    return m, out


def map_png(name, kd_key, rep):
    """Returns the png path of the splice map for dataset `name`, knockdown `kd_key` and replicate `rep`."""
    return MAP_DIR / "{}_KD".format(kd_key) / "{}_{}.se.png".format(name, rep)


def finished_outputs(name, kd_key, rep):
    """Returns every file a permutation-test map of dataset `name`, knockdown `kd_key` and replicate `rep` writes, if all
    exist and are newer than its event lists; otherwise None (always None with the z-score test or
    RBPMAPS_RECOMPUTE=1).
    """
    out = map_png(name, kd_key, rep)
    files = event_files[kd_key]
    base = str(out)[:-len(".png")]
    need = [out] + [Path("{}.{}.{}".format(base, files[s].name, kind)) for s in SETS for kind in ("means.txt", "normed_matrix.txt")]
    need += [Path("{}.{}.{}.randsample.tsv".format(out, files[SETS[2]].name, files[s].name)) for s in SETS[:2]]
    newest_list = max(files[s].stat().st_mtime for s in SETS)
    if SIGTEST != "permutation" or os.environ.get("RBPMAPS_RECOMPUTE") == "1":
        return None
    if all(p.exists() and p.stat().st_mtime > newest_list for p in need):
        return need
    return None


def load_map(name, kd_key, rep):
    """Reads a finished map (dataset `name`, knockdown `kd_key`, replicate `rep`) back from disk: the normalized
    matrices, mean lines and background band that sections 5-6 use; returns it and its png.
    """
    from types import SimpleNamespace
    out = map_png(name, kd_key, rep)
    files = event_files[kd_key]
    base = str(out)[:-len(".png")]
    lo = hi = None
    for s in SETS[:2]:
        rs = pd.read_csv("{}.{}.{}.randsample.tsv".format(out, files[SETS[2]].name, files[s].name), sep="\t", index_col=0)
        b, t = norm.bottom_top_values_from_dataframe(rs, 0.5, 0.5)
        lo = np.array(b, float) if lo is None else np.fmin(lo, np.array(b, float))
        hi = np.array(t, float) if hi is None else np.fmax(hi, np.array(t, float))
    m = SimpleNamespace(norm_matrices={}, lines=[])
    for s in SETS:
        key = str(files[s])
        m.norm_matrices[key] = pd.read_csv("{}.{}.normed_matrix.txt".format(base, files[s].name), index_col=0)
        means = np.loadtxt("{}.{}.means.txt".format(base, files[s].name))
        band = (lo, hi) if s == SETS[2] else (means, means)
        m.lines.append(SimpleNamespace(annotation_src_file=key, means=means, error_neg=band[0], error_pos=band[1]))
    return m, out


PAIRS = U.MAP_PAIRS
maps = {}
for name, kd_key in PAIRS:
    for rep in ("rep1", "rep2"):
        if finished_outputs(name, kd_key, rep):
            maps[(name, kd_key, rep)], png = load_map(name, kd_key, rep)
            print(name, "x", kd_key, "knockdown", rep, "-> reused (finished after the event lists were made):", png, flush=True)
            continue
        maps[(name, kd_key, rep)], png = splice_map(name, kd_key, rep)
        print(name, "x", kd_key, "knockdown", rep, "->", png, flush=True)
        display(Image(str(png), width=900))

## 5. PICALM's own traces

A skipped-exon matrix has 1,400 columns: four 350-nt regions (upstream exon end, the cassette exon's 3′ splice site, its 5′ splice site, the downstream exon start). Columns 350-699 cover the cassette 3′ splice site (300 nt of intron, then 50 nt of exon); columns 700-1049 the 5′ splice site (50 nt of exon, then 300 nt of intron). PICALM exon 13 is `chr11:85,990,249-85,990,399` (GRCh38, 0-based start, minus strand); the 24-nt exon `chr11:85,978,069-85,978,093`.

The cell below takes each PICALM exon's row of the normalized matrix, wherever it falls (included, excluded or background), and reports the peak in the region that RBP's rule points to: upstream of the 3′ splice site for exon 13 (window maps and the RBP-ARK windows put QKI 0-100 nt upstream) and downstream of the 24-nt exon's 5′ splice site.

In [ ]:
X3 = np.arange(-300, 50)    # cassette 3' splice site region: nt from the 3'ss (negative = intron)
X5 = np.arange(-49, 301)    # cassette 5' splice site region: nt from the 5'ss (positive = intron)
REGION = {("QKI", "exon13"): "3ss", ("QKI", "exon24nt"): "5ss"}


def picalm_row(m, kd_key, exon):
    """Returns the event set holding PICALM `exon` and its row of map `m`'s normalized matrix (knockdown `kd_key`), or
    (None, None).
    """
    c, s, a, b = U.PICALM_GRCH38[exon]
    for set_name in SETS:
        matrix = m.norm_matrices[str(event_files[kd_key][set_name])]
        hits = [i for i in matrix.index if tuple(i.split("\t")[3:7]) == (c, s, str(a), str(b))]
        if hits:
            return set_name, np.asarray(matrix.loc[hits[0]], dtype=float)
    return None, None


rows, traces, lines_out = [], {}, []
for (name, kd_key, rep), m in maps.items():
    rbp = U.ECLIP[name]["rbp"]
    for exon in U.PICALM_GRCH38:
        set_name, t = picalm_row(m, kd_key, exon)
        if t is None:
            continue
        traces[(name, kd_key, rep, exon)] = t
        region = REGION[(rbp, exon)]
        seg, x = (t[350:700], X3) if region == "3ss" else (t[700:1050], X5)
        rows.append({"dataset": name, "knockdown": kd_key, "rep": rep, "exon": exon, "set": set_name, "region": region,
                     "peak_nt": int(x[np.nanargmax(seg)]), "max_signal_x1e4": round(float(np.nanmax(seg)) * 1e4, 1),
                     "signal_0_100nt_x1e4": round(float(np.nansum(seg[(x >= -100) & (x < 0)] if region == "3ss"
                                                                    else seg[(x > 0) & (x <= 100)])) * 1e4, 1)})
        np.savetxt(MAP_DIR / "{}_KD".format(kd_key) / "PICALM_{}_{}_{}.txt".format(exon, name, rep), t)
    files = {str(v): k for k, v in event_files[kd_key].items()}
    for line in m.lines:
        set_name = files.get(line.annotation_src_file)
        if set_name:
            lines_out.append([name, kd_key, rep, set_name, *np.asarray(line.means, dtype=float)])
picalm_traces = pd.DataFrame(rows)
picalm_traces.to_csv(ROOT / "results" / "rbp_maps_picalm_traces_{}.tsv".format(SOURCE), sep="\t", index=False)
pd.DataFrame(lines_out).to_csv(ROOT / "figures" / "data" / "rbp_maps_lines_{}.tsv.gz".format(SOURCE), sep="\t",
                               index=False, header=False)
picalm_traces

In [ ]:
import matplotlib as mpl
import seaborn as sns

STYLE = {"included-upon-knockdown": ("#eb6834", "more included after knockdown"),
         "excluded-upon-knockdown": ("#2a78d6", "more excluded after knockdown"),
         "native-cassette-background": ("#8a8984", "native cassette exons")}
TRACE = {"QKI": ("#7a3db8", "#4b2172")}
FIG_MAPS = [p for p in U.MAP_PAIRS if (p[0], p[1], "rep1") in maps]


def plain_style():
    """Resets matplotlib after rbp-maps' large seaborn context, for this notebook's own figure."""
    mpl.rcdefaults()
    sns.set_theme(context="paper", style="ticks", rc={"xtick.labelsize": 7, "ytick.labelsize": 7})


def panel(name, kd_key, rep, ax_regions, ax_trace):
    """Draws one splice map (dataset `name`, knockdown `kd_key`, replicate `rep`) on the four axes `ax_regions`, and
    PICALM exon 13's own trace on `ax_trace`.
    """
    m = maps[(name, kd_key, rep)]
    rbp = U.ECLIP[name]["rbp"]
    files = {str(v): k for k, v in event_files[kd_key].items()}
    titles = ["upstream exon | intron", "intron | cassette exon (3′ss)", "cassette exon | intron (5′ss)",
              "intron | downstream exon"]
    for line in m.lines:
        set_name = files.get(line.annotation_src_file)
        if set_name is None:
            continue
        color, label = STYLE[set_name]
        means = np.asarray(line.means, dtype=float)
        for i, ax in enumerate(ax_regions):
            seg = slice(i * 350, (i + 1) * 350)
            if set_name == "native-cassette-background" and getattr(line, "error_pos", None) is not None:
                ax.fill_between(np.arange(350), np.asarray(line.error_neg, dtype=float)[seg] * 1e4,
                                np.asarray(line.error_pos, dtype=float)[seg] * 1e4, color=color, alpha=0.25, lw=0)
            ax.plot(np.arange(350), means[seg] * 1e4, color=color, lw=1, label=label if i == 0 else None)
    for ax, title in zip(ax_regions, titles):
        ax.set_title(title, fontsize=8)
        ax.set_xticks([])
        ax.spines[["top", "right"]].set_visible(False)
    ax_regions[0].set_ylabel("{} eCLIP\n{} KD, {}\nIP − input (×10⁻⁴)".format(name, kd_key.split("_")[1], rep), fontsize=7)
    t = traces.get((name, kd_key, rep, "exon13"))
    if t is None:
        ax_trace.set_axis_off()
        return
    region = REGION[(rbp, "exon13")]
    seg, x = (t[350:700], X3) if region == "3ss" else (t[700:1050], X5)
    seg = seg * 1e4
    fill, edge = TRACE[rbp]
    ax_trace.fill_between(x, 0, np.clip(seg, 0, None), color=fill, alpha=0.35, lw=0)
    ax_trace.plot(x, seg, color=edge, lw=0.8)
    peak = x[np.nanargmax(seg)]
    ax_trace.axvline(peak, color=edge, ls="--", lw=0.7)
    ax_trace.text(0.02, 1.02, "PICALM exon 13 · peak {:+d} nt".format(int(peak)), transform=ax_trace.transAxes, fontsize=7)
    ax_trace.set_xlim(x[0], x[-1])
    ax_trace.set_xlabel("nt from cassette {} splice site".format("3′" if region == "3ss" else "5′"), fontsize=7)
    ax_trace.spines[["top", "right"]].set_visible(False)


plain_style()
fig = plt.figure(figsize=(12, 3.4 * len(FIG_MAPS)))
grid = fig.add_gridspec(2 * len(FIG_MAPS), 4, height_ratios=[2.2, 1] * len(FIG_MAPS), hspace=0.75, wspace=0.08,
                       top=1 - 0.5 / (3.4 * len(FIG_MAPS)))
for k, (name, kd_key) in enumerate(FIG_MAPS):
    first = fig.add_subplot(grid[2 * k, 0])
    regions = [first] + [fig.add_subplot(grid[2 * k, j], sharey=first) for j in range(1, 4)]
    for ax in regions[1:]:
        ax.tick_params(labelleft=False)
    trace_col = 1 if REGION[(U.ECLIP[name]["rbp"], "exon13")] == "3ss" else 2
    panel(name, kd_key, "rep1", regions, fig.add_subplot(grid[2 * k + 1, trace_col]))
fig.axes[0].legend(frameon=False, fontsize=7, loc="upper left")
fig.suptitle("Panels d, e · QKI splice maps (RBP-ARK BAMs, replicate 1) with PICALM exon 13's own trace · "
             "rMATS source: " + SOURCE, fontsize=10, y=1 - 0.15 / (3.4 * len(FIG_MAPS)))
for ext in ("pdf", "png"):
    fig.savefig(FIG_DIR / "panel_de_rbp_maps_{}.{}".format(SOURCE, ext), dpi=300, bbox_inches="tight")
plt.show()

**Reading the result.** QKI: the orange line (exons QKI represses) should rise in the last ~100 nt of intron before the cassette exon's 3′ splice site, and the blue line (exons QKI activates) just after the 5′ splice site, both above the background band. PICALM exon 13's QKI trace should peak in the same upstream window. All four maps are in `<data>/rbp_maps/maps_<source>/`.

**Notes.**

- **Strand.** If a map looks like noise, check `sense_fraction` in section 3. Read-2 eCLIP reads should be well above 0.5.
- **Outliers.** `conf=0.95` drops the most extreme 5% of events at each position before averaging the lines. The PICALM traces come from the matrix before that step.